In [1]:
import numpy as np
from scipy import stats
from pprint import pprint
import json
abl_results = json.load(open("C:/users/nicol/master_thesis/code/analysis/pp_finetune_results.json"))
import os
os.chdir("C:/users/nicol/master_thesis/code")
from src.eval.calculate_dhgp import get_correlations
import numpy as np
def round_values(data, decimals=4):
    if isinstance(data, dict):
        return {k: round_values(v, decimals) for k, v in data.items()}
    elif isinstance(data, float) or isinstance(data, int):
        return round(data, decimals)
    else:
        return data
total_results = []

for entry in abl_results:
    config_name = entry["config"]
    experiment_results = {
        "config": config_name,
        "splits": {},
        "t_test": {}
    }
    
    ssl_data = entry["ssl_results"]           # Format: {"split1": {id: score...}, ...}
    sup_data = entry["supervised_results"]    # Format: {"split1": {id: score...}, ...}
    
    ssl_corrs = []
    sup_corrs = []
    
    all_mae_ssl = []
    all_mae_sup =[]
    # We assume splits are labeled 'split1' through 'split5'
    split_keys = sorted(ssl_data.keys()) 
    
    for split_key in split_keys:
        # 1. Calculate SSL Correlation
        ssl_pred_dict = ssl_data.get(split_key, {})
        ssl_res = get_correlations(prediction_dict=ssl_pred_dict, type="spearman")
        ssl_corr = ssl_res.get("correlation", 0.0)

        mae_ssl = ssl_res.get("mean_absolute_error")
        all_mae_ssl.append(mae_ssl)

        if ssl_corr == "NAN" or (isinstance(ssl_corr, float) and np.isnan(ssl_corr)):
            ssl_corr = 0.0
            
        # 2. Calculate Supervised Correlation
        sup_pred_dict = sup_data.get(split_key, {})
        sup_res = get_correlations(prediction_dict=sup_pred_dict, type="spearman")
        sup_corr = sup_res.get("correlation", 0.0)

        mae_sup = sup_res.get("mean_absolute_error")
        all_mae_sup.append(mae_sup)
        if sup_corr == "NAN" or (isinstance(sup_corr, float) and np.isnan(sup_corr)):
            sup_corr = 0.0
            
        # Store for the t-test
        ssl_corrs.append(ssl_corr)
        sup_corrs.append(sup_corr)
        
        # Save per-split values
        experiment_results["splits"][split_key] = {
            "ssl_correlation": round_values(ssl_corr),
            "sup_correlation": round_values(sup_corr),
            "mae_sup":mae_sup,
            "mae_ssl":mae_ssl
        }

    # 3. Perform Paired T-Test (SSL vs Supervised)
    # This compares the two lists of 5 correlations
    t_stat, p_value = stats.ttest_rel(ssl_corrs, sup_corrs, alternative="greater")
    
    experiment_results["summary"] = {
        "mean_ssl_corr": round_values(np.mean(ssl_corrs)),
        "mean_sup_corr": round_values(np.mean(sup_corrs)),
        "t_statistic": round_values(t_stat),
        "p_value": round_values(p_value),
        "significant_05": p_value < 0.05,
        "mae_sup": round_values(np.mean(all_mae_sup)),
        "mae_ssl":round_values(np.mean(all_mae_ssl)),
        "mae_sup_std":round_values(np.std(all_mae_sup)),
        "mae_ssl_std":round_values(np.std(all_mae_ssl)),
    }

    total_results.append(experiment_results)

pprint(total_results)

[{'config': 'A',
  'splits': {'split1': {'mae_ssl': 21.287881117944515,
                        'mae_sup': 20.616348546113176,
                        'ssl_correlation': 0.7516,
                        'sup_correlation': 0.7701},
             'split2': {'mae_ssl': 20.24948255247448,
                        'mae_sup': 23.006234994331606,
                        'ssl_correlation': 0.8005,
                        'sup_correlation': 0.7383},
             'split3': {'mae_ssl': 20.35972284026777,
                        'mae_sup': 23.36213435862698,
                        'ssl_correlation': 0.7786,
                        'sup_correlation': 0.7263},
             'split4': {'mae_ssl': 20.01947895306501,
                        'mae_sup': 21.757723273206405,
                        'ssl_correlation': 0.7783,
                        'sup_correlation': 0.7628},
             'split5': {'mae_ssl': 19.589258052168397,
                        'mae_sup': 23.249982770973208,
                        '

C:\Users\nicol\AppData\Local\Temp\ipykernel_26704\4281586237.py:74: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  t_stat, p_value = stats.ttest_rel(ssl_corrs, sup_corrs, alternative="greater")
c:\Users\nicol\master_thesis\code\.venv\Lib\site-packages\numpy\core\fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\nicol\master_thesis\code\.venv\Lib\site-packages\numpy\core\_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
c:\Users\nicol\master_thesis\code\.venv\Lib\site-packages\numpy\core\_methods.py:206: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
c:\Users\nicol\master_thesis\code\.venv\Lib\site-packages\numpy\core\_methods.py:163: RuntimeWarning: invalid value encountered in divide
  arrmean = u

## Non Parametric Version (Wilcoxon Signed-Rank Test)

In [2]:
import numpy as np
from scipy import stats
from pprint import pprint
import json
abl_results = json.load(open("C:/users/nicol/master_thesis/code/analysis/pp_finetune_results.json"))
import os
os.chdir("C:/users/nicol/master_thesis/code")
from src.eval.calculate_dhgp import get_correlations
import numpy as np
def round_values(data, decimals=4):
    if isinstance(data, dict):
        return {k: round_values(v, decimals) for k, v in data.items()}
    elif isinstance(data, float) or isinstance(data, int):
        return round(data, decimals)
    else:
        return data
abl_results = json.load(open("C:/users/nicol/master_thesis/code/analysis/pp_finetune_results.json"))

total_results = []

for entry in abl_results:
    config_name = entry["config"]
    # Update dict structure to reflect non-parametric test
    experiment_results = {
        "config": config_name,
        "splits": {},
        "wilcoxon_test": {} 
    }
    
    ssl_data = entry["ssl_results"]           
    sup_data = entry["supervised_results"]    
    
    ssl_corrs = []
    sup_corrs = []
    
    all_mae_ssl = []
    all_mae_sup = []
    
    split_keys = sorted(ssl_data.keys()) 
    
    for split_key in split_keys:
        # 1. Calculate SSL Correlation
        ssl_pred_dict = ssl_data.get(split_key, {})
        ssl_res = get_correlations(prediction_dict=ssl_pred_dict, type="spearman")
        ssl_corr = ssl_res.get("correlation", 0.0)

        mae_ssl = ssl_res.get("mean_absolute_error")
        all_mae_ssl.append(mae_ssl)

        if ssl_corr == "NAN" or (isinstance(ssl_corr, float) and np.isnan(ssl_corr)):
            ssl_corr = 0.0
            
        # 2. Calculate Supervised Correlation
        sup_pred_dict = sup_data.get(split_key, {})
        sup_res = get_correlations(prediction_dict=sup_pred_dict, type="spearman")
        sup_corr = sup_res.get("correlation", 0.0)

        mae_sup = sup_res.get("mean_absolute_error")
        all_mae_sup.append(mae_sup)
        if sup_corr == "NAN" or (isinstance(sup_corr, float) and np.isnan(sup_corr)):
            sup_corr = 0.0
            
        # Store for the test
        ssl_corrs.append(ssl_corr)
        sup_corrs.append(sup_corr)
        
        # Save per-split values
        experiment_results["splits"][split_key] = {
            "ssl_correlation": round_values(ssl_corr),
            "sup_correlation": round_values(sup_corr),
            "mae_sup": mae_sup,
            "mae_ssl": mae_ssl
        }

    # ---------------------------------------------------------
    # 3. Perform Wilcoxon Signed-Rank Test (Non-Parametric)
    # ---------------------------------------------------------
    # 'alternative="greater"' tests if ssl_corrs is stochastically greater than sup_corrs
    # Note: For N=5, exact p-values are calculated.
    try:
        w_stat, p_value = stats.wilcoxon(ssl_corrs, sup_corrs, alternative="greater")
    except ValueError:
        print("dab")
        # Handle case where all differences are zero (rare but possible)
        w_stat, p_value = 0.0, 1.0

    experiment_results["summary"] = {
        "mean_ssl_corr": round_values(np.mean(ssl_corrs)),
        "mean_sup_corr": round_values(np.mean(sup_corrs)),
        "wilcoxon_statistic": round_values(w_stat),
        "p_value": round_values(p_value),
        "significant_05": bool(p_value < 0.05),
        "mae_sup": round_values(np.mean(all_mae_sup)),
        "mae_ssl": round_values(np.mean(all_mae_ssl)),
        "mae_sup_std": round_values(np.std(all_mae_sup)),
        "mae_ssl_std": round_values(np.std(all_mae_ssl)),
    }

    total_results.append(experiment_results)

pprint(total_results)

dab
dab
[{'config': 'A',
  'splits': {'split1': {'mae_ssl': 21.287881117944515,
                        'mae_sup': 20.616348546113176,
                        'ssl_correlation': 0.7516,
                        'sup_correlation': 0.7701},
             'split2': {'mae_ssl': 20.24948255247448,
                        'mae_sup': 23.006234994331606,
                        'ssl_correlation': 0.8005,
                        'sup_correlation': 0.7383},
             'split3': {'mae_ssl': 20.35972284026777,
                        'mae_sup': 23.36213435862698,
                        'ssl_correlation': 0.7786,
                        'sup_correlation': 0.7263},
             'split4': {'mae_ssl': 20.01947895306501,
                        'mae_sup': 21.757723273206405,
                        'ssl_correlation': 0.7783,
                        'sup_correlation': 0.7628},
             'split5': {'mae_ssl': 19.589258052168397,
                        'mae_sup': 23.249982770973208,
                 

c:\Users\nicol\master_thesis\code\.venv\Lib\site-packages\scipy\_lib\_util.py:798: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  return fun(*args, **kwargs)
c:\Users\nicol\master_thesis\code\.venv\Lib\site-packages\numpy\core\fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\nicol\master_thesis\code\.venv\Lib\site-packages\numpy\core\_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
c:\Users\nicol\master_thesis\code\.venv\Lib\site-packages\numpy\core\_methods.py:206: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
c:\Users\nicol\master_thesis\code\.venv\Lib\site-packages\numpy\core\_methods.py:163: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrm

In [1]:
import json
from pprint import pprint
import ast
print(json.dumps(ast.literal_eval(open('./temp.txt').read().strip()), indent=4))

{
    "split1": {
        "D00-41129-IIIA": 0.4162839248434238,
        "D00-42686-IIIA": 0.6580549168132017,
        "D01-40292-IID": 0.35551790900290414,
        "D01-43315-IIIA": 0.4686439612919442,
        "D02-41143-IIIB": 0.034535452322738386,
        "D03-40748-IA": 0.3945740123750595,
        "D03-41644-A": 0.7746102449888641,
        "D03-42154-IIB": 0.32079553384508025,
        "D04-40087-B": 0.929000546149645,
        "D05-40899-IIIA": 0.7791563275434243,
        "D05-40899-IIID": 0.945360824742268,
        "D05-41278-IIE": 0.12516297262059975,
        "D05-41433-IIIA": 0.11147298297557365,
        "D05-41487-IIB": 0.1823222987733154,
        "D05-42937-IIIA": 0.7066748315982854,
        "D06-42469-IIA": 0.31892812887236677,
        "D06-43105-IIA": 0.5477685950413224,
        "D06-43386-IIB": 0.22512360939431397,
        "D07-40172-IA": 0.4696479158235532,
        "D07-40758-IIB": 0.28775055679287304,
        "D07-42377-IIB": 1.0,
        "D08-40122-D": 0.8012354152367879,


In [ ]:
labeled_ids = [
            "H18-24578-IIB",
            "H15-15551-IIB",
            "D00-41129-IIIC",
            "H18-16958-IIA",
            "D01-41285-IIIB",
            "H18-15067-A",
            "H01-9856-C",
            "H17-4726-IIA",
            "H16-5021-IIC",
            "H18-9736-B",
            "H16-5021-IIA",
            "D02-40504-IIA",
            "H16-12643-IA",
            "H18-12073-IIA",
            "D09-42190-E",
            "D09-42190-A",
            "H18-20317-IIA",
            "H15-16121-IA",
            "H16-17142-II",
            "D10-41545-A",
            "H14-12726-IIA",
            "H18-10906-IIIA",
            "H15-7933-IVC",
            "H18-16599-IA",
            "H18-24578-IIA",
            "H18-11547-IIA",
            "H18-20690-A",
            "H16-23157-IIA",
            "H16-12870-IIB",
            "D02-40593-IIIA",
            "D02-40504-IIIB",
            "D00-40067-IIA",
            "D00-41129-IIIA",
            "H18-23553-IA",
            "H18-12073-IIIB",
            "H18-2220-IA",
            "D00-42686-IIIA",
            "D01-43315-IIIA",
            "H18-4312-IVA",
            "H18-14189-IIB"
]

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from imageio.v3 import imread

plt.rcParams["font.family"] = "serif"
Image.MAX_IMAGE_PIXELS = None

os.chdir("C:/users/nicol/master_thesis/code")

from src.eval.calc_dice import geojson_to_mask_corrected
from src.eval.calculate_dhgp import calculate_dhgp_faber

CONFIG = "I"
SPLIT = "split3"

BASE_RESULTS_DIR_SUP = "D:/tfm_data/hpc_results/ablationstudy_results/final_supervised_dice_extended_pp_edited_25_12_31_1810"
BASE_RESULTS_DIR_SSL = "D:/tfm_data/hpc_results/ablationstudy_results/final_ssl_dice_extended_pp_edited_25_12_31_1811"
THUMBNAIL_DIR = "D:/tfm_data/10mpp_ds"

COLOR_MAP_NP = np.array(
    [
        [0, 0, 0],
        [255, 0, 0],
        [0, 255, 0],
        [0, 0, 255],
    ],
    dtype=np.uint8,
)

def map_seg_to_color_np(seg_np):
    return COLOR_MAP_NP[seg_np.astype(np.int64)]

with open("./analysis/pp_finetune_results.json", "r") as f:
    results_json = json.load(f)

for entry in results_json:
    if entry["config"] == CONFIG:
        ssl_results = entry["ssl_results"][SPLIT]
        sup_results = entry["supervised_results"][SPLIT]
        break
else:
    raise ValueError("Config not found")

common_wsis = sorted(set(ssl_results.keys()) & set(sup_results.keys()))

plotting_data = {}

for wsi_id in common_wsis:
    is_labeled = wsi_id in labeled_ids
    if is_labeled:
        mask_sup = map_seg_to_color_np(
            imread(
                f"{BASE_RESULTS_DIR_SUP}/{SPLIT}/{wsi_id}_postprocessed.png"
            )
        )

        mask_ssl = map_seg_to_color_np(
            imread(
                f"{BASE_RESULTS_DIR_SSL}/{SPLIT}/{wsi_id}_postprocessed.png"
            )
        )

        gt_mask = map_seg_to_color_np(
            geojson_to_mask_corrected(wsi_id, scale=128)
        )

        thumbnail = Image.open(f"{THUMBNAIL_DIR}/{wsi_id}.png")

        dhgp_sup = sup_results[wsi_id] * 100
        dhgp_ssl = ssl_results[wsi_id] * 100
        dhgp_gt = calculate_dhgp_faber(
            geojson_to_mask_corrected(wsi_id, scale=128)
        ) * 100

        plotting_data[wsi_id] = {
            "sup": mask_sup,
            "ssl": mask_ssl,
            "gt": gt_mask,
            "thumb": thumbnail,
            "dhgp_sup": dhgp_sup,
            "dhgp_ssl": dhgp_ssl,
            "dhgp_gt": dhgp_gt,
            "labeled": is_labeled,
        }

for wsi_id, data in plotting_data.items():
    fig, axes = plt.subplots(1, 4, figsize=(15, 8))

    fig.suptitle(
        f"WSI: {wsi_id} | "
        f"Supervised: {data['dhgp_sup']:.1f} | "
        f"Semi-Supervised: {data['dhgp_ssl']:.1f} | "
        f"Mask GT: {data['dhgp_gt']:.1f} | "
        f"{'Labeled' if data['labeled'] else 'Unlabeled'}",
        fontsize=16,
        y=0.93,
    )

    axes[0].imshow(data["sup"])
    axes[0].set_title("Supervised")
    axes[0].axis("off")

    axes[1].imshow(data["ssl"])
    axes[1].set_title("Semi-Supervised")
    axes[1].axis("off")

    axes[2].imshow(data["gt"])
    axes[2].set_title("Ground Truth")
    axes[2].axis("off")

    axes[3].imshow(data["thumb"])
    axes[3].set_title("Thumbnail")
    axes[3].axis("off")

    plt.tight_layout(rect=[0, 0.03, 1, 0.88])
    plt.show()


In [ ]:
import numpy as np
import pandas as pd
import json
abl_results = json.load(open("C:/users/nicol/master_thesis/code/analysis/pp_finetune_results.json"))
total_results = []
def round_values(data, decimals=4):
    if isinstance(data, dict):
        return {k: round_values(v, decimals) for k, v in data.items()}
    elif isinstance(data, float) or isinstance(data, int):
        return round(data, decimals)
    else:
        return data
def calculate_mae(merged_df):
    """Calculates MAE by scaling predicted_dHGP (0-1.0) to the dHGP scale (0-100)."""
    # Scale prediction: y_pred_scaled = predicted_dHGP * 100
    y_pred_scaled = merged_df['predicted_dHGP'].apply(lambda x: float(x) * 100.0)
    y_true = merged_df['dHGP']
    
    # Calculate Mean Absolute Error (MAE)
    mae = np.mean(np.abs(y_true - y_pred_scaled))
    std = np.std(np.abs(y_true - y_pred_scaled))
    return mae,std
def get_correlations(
    prediction_dict: dict,
    model_name="Test",
    type="pearson",
    gt_filter=None  # <-- NEW
):
    dhgp_scores = pd.read_csv(
        "./src/config/dhgpscores_allslides.csv",
        encoding="latin-1",
        sep=";"
    )

    emc_scores = dhgp_scores[
        (dhgp_scores['Cohort'] == 'EMC') &
        (dhgp_scores['Tissue'] == 'CRLM') &
        (
            (dhgp_scores['Resection'] == 'PA_nummer_CRLM') |
            (dhgp_scores['Resection'] == 'Stage2_PA_nummer_CRLM')
        )
    ][['EMC_flnm', 'dHGP']]

    emc_scores = emc_scores.dropna(subset=['dHGP'])
    emc_scores = emc_scores[
        emc_scores['dHGP'].astype(str).str.strip().isin(['-', '']) == False
    ]
    emc_scores['dHGP'] = pd.to_numeric(emc_scores['dHGP'], errors='coerce')

    # ---- Apply GT FILTER ----
    if gt_filter == "gt_0":
        emc_scores = emc_scores[emc_scores['dHGP'] == 0]
    elif gt_filter == "gt_1_99":
        emc_scores = emc_scores[(emc_scores['dHGP'] >= 1) & (emc_scores['dHGP'] <= 99)]
    elif gt_filter == "gt_100":
        emc_scores = emc_scores[emc_scores['dHGP'] == 100]

    # Keep only predicted WSIs
    wsi_ids = list(prediction_dict.keys())
    target_rows = emc_scores[emc_scores['EMC_flnm'].isin(wsi_ids)]

    merged = target_rows.copy()
    merged['predicted_dHGP'] = merged['EMC_flnm'].map(prediction_dict)
    merged = merged.dropna(subset=['predicted_dHGP', 'dHGP'])

    if len(merged) < 2 or merged['predicted_dHGP'].std() == 0 or merged['dHGP'].std() == 0:
        correlation = float('nan')
    else:
        correlation = merged['predicted_dHGP'].corr(
            merged['dHGP'], method=type
        )

    mean_absolute_error, std = calculate_mae(merged)

    return {
        "model": model_name,
        "correlation": correlation,
        "mean_absolute_error": mean_absolute_error,
        "mae_std": std,
        "n_samples": len(merged)
    }

GT_BUCKETS = {
    "gt_0": "GT = 0",
    "gt_1_99": "GT ∈ [1, 99]",
    "gt_100": "GT = 100"
}

for entry in abl_results:
    config_name = entry["config"]

    experiment_results = {
        "config": config_name,
        "splits": {},
        "summary": {}
    }

    ssl_data = entry["ssl_results"]
    sup_data = entry["supervised_results"]

    split_keys = sorted(ssl_data.keys())

    for split_key in split_keys:
        experiment_results["splits"][split_key] = {}

        ssl_pred_dict = ssl_data.get(split_key, {})
        sup_pred_dict = sup_data.get(split_key, {})

        for bucket_key, bucket_name in GT_BUCKETS.items():
            ssl_res = get_correlations(
                ssl_pred_dict,
                type="spearman",
                gt_filter=bucket_key
            )

            sup_res = get_correlations(
                sup_pred_dict,
                type="spearman",
                gt_filter=bucket_key
            )

            ssl_corr = ssl_res["correlation"]
            sup_corr = sup_res["correlation"]

            if isinstance(ssl_corr, float) and np.isnan(ssl_corr):
                ssl_corr = 0.0
            if isinstance(sup_corr, float) and np.isnan(sup_corr):
                sup_corr = 0.0

            experiment_results["splits"][split_key][bucket_key] = {
                "ssl_corr": round_values(ssl_corr),
                "sup_corr": round_values(sup_corr),
                "mae_ssl": round_values(ssl_res["mean_absolute_error"]),
                "mae_sup": round_values(sup_res["mean_absolute_error"]),
                "n_samples": ssl_res["n_samples"]
            }

    total_results.append(experiment_results)


In [ ]:
total_results